In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, identity
from scipy.sparse.linalg import inv
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from collections import defaultdict
import time
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, identity
from scipy.sparse.linalg import inv
from sklearn.preprocessing import LabelEncoder
import time
from tqdm.auto import tqdm

In [2]:
user_item_ratings_subratings = pd.read_csv('../data/filtered/hotelrec_filtered_00001.csv')
ratings_df = user_item_ratings_subratings[['user_id', 'item_id', 'rating']]
ratings_df = ratings_df.sample(n=100_000, random_state=42)
# drop duplicates
print(f"Original shape: {ratings_df.shape}")
ratings_df = ratings_df.drop_duplicates(subset=['user_id', 'item_id'])
print(f"Shape after dropping duplicates: {ratings_df.shape}")

# load data as half-precision floats and integers as half ints to save memory
ratings_df['rating'] = ratings_df['rating'].astype(np.float16)
ratings_df['user_id'] = ratings_df['user_id'].astype(np.int32)
ratings_df['item_id'] = ratings_df['item_id'].astype(np.int32)
ratings_df.info()

Original shape: (100000, 3)
Shape after dropping duplicates: (99003, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 99003 entries, 4647116 to 19020529
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   user_id  99003 non-null  int32  
 1   item_id  99003 non-null  int32  
 2   rating   99003 non-null  float16
dtypes: float16(1), int32(2)
memory usage: 1.7 MB


In [3]:
# --- Provided / OLD EASE Class ---
class EASE:
    def __init__(self):
        self.user_enc = LabelEncoder()
        self.item_enc = LabelEncoder()

    def _get_users_and_items(self, df):
        users = self.user_enc.fit_transform(df.loc[:, 'user_id'])
        items = self.item_enc.fit_transform(df.loc[:, 'item_id'])
        return users, items

    def fit(self, df, lambda_: float = 0.5, implicit=True):
        """
        df: pandas.DataFrame with columns user_id, item_id and (rating)
        lambda_: l2-regularization term
        implicit: if True, ratings are ignored and taken as 1, else normalized ratings are used
        """
        users, items = self._get_users_and_items(df)
        values = (
            np.ones(df.shape[0])
            if implicit
            else df['rating'].to_numpy() / df['rating'].max()
        )

        X = csr_matrix((values, (users, items)))
        self.X = X

        G = X.T.dot(X).toarray()
        diagIndices = np.diag_indices(G.shape[0])
        G[diagIndices] += lambda_
        P = np.linalg.inv(G)
        B = P / (-np.diag(P))
        B[diagIndices] = 0

        self.B = B
        self.pred = X.dot(B)

    def predict(self, train, users, items, k):
        items = self.item_enc.transform(items)
        dd = train.loc[train.user_id.isin(users)]
        dd['ci'] = self.item_enc.transform(dd.item_id)
        dd['cu'] = self.user_enc.transform(dd.user_id)
        g = dd.groupby('cu')
        with Pool(cpu_count()) as p:
            user_preds = p.starmap(
                self.predict_for_user,
                [(user, group, self.pred[user, :], items, k) for user, group in g],
            )
        df = pd.concat(user_preds)
        df['item_id'] = self.item_enc.inverse_transform(df['item_id'])
        df['user_id'] = self.user_enc.inverse_transform(df['user_id'])
        return df

    @staticmethod
    def predict_for_user(user, group, pred, items, k):
        watched = set(group['ci'])
        candidates = [item for item in items if item not in watched]
        pred = np.take(pred, candidates)
        res = np.argpartition(pred, -k)[-k:]
        r = pd.DataFrame(
            {
                "user_id": [user] * len(res),
                "item_id": np.take(candidates, res),
                "score": np.take(pred, res),
            }
        ).sort_values('score', ascending=False)
        return r

In [3]:
# --- Optimmized EASE ---- 
class EASE:
    def __init__(self):
        self.user_enc = LabelEncoder()
        self.item_enc = LabelEncoder()
        self.B = None # This will be our item-item similarity matrix
        self.X = None # This will be our user-item training matrix

    def fit(self, df, lambda_: float = 100, implicit=True):
        """
        Trains the EASE model.
        
        Key Optimizations:
        1. All matrix operations are kept in sparse format to conserve memory.
        2. Uses scipy's sparse inverse, which is much faster and more memory-efficient.
        3. Avoids pre-calculating all predictions, saving massive amounts of memory.
        """
        print("Fitting EASE model...")
        
        # --- Data Encoding ---
        df['encoded_user'] = self.user_enc.fit_transform(df['user_id'])
        df['encoded_item'] = self.item_enc.fit_transform(df['item_id'])
        
        # --- User-Item Matrix Creation ---
        values = np.ones(df.shape[0]) if implicit else df['rating'].to_numpy() / df['rating'].max()
        self.X = csr_matrix((values, (df['encoded_user'], df['encoded_item'])))
        
        # --- Gram Matrix and Inversion (Memory & Speed Optimized) ---
        # The Gram matrix G is computed and kept as a sparse matrix (csc_matrix is efficient for this)
        G = self.X.T.dot(self.X).tocsc()
        
        # Add the regularization term to the diagonal
        G += identity(G.shape[0]) * lambda_
        
        # Invert the sparse matrix. This is the most compute-intensive part.
        P = inv(G)
        
        # Convert to a dense array ONLY for the final division and diagonal manipulation
        P_dense = P.toarray()
        
        # Calculate the final item-item similarity matrix B
        # The diagonal of P contains the negative inverse of the item "self-similarity"
        B = P_dense / (-np.diag(P_dense))
        np.fill_diagonal(B, 0)
        
        self.B = B.astype(np.float16) # Store as float32 to save memory
        print("Fit complete.")

    def predict(self, train_df, users, k, batch_size=2048):
        """
        Generates top-k recommendations for a list of users.
        
        Key Optimizations:
        1. NO MULTIPROCESSING: Uses a fast, vectorized batch processing approach.
        2. Batched Prediction: Calculates scores in small batches to keep memory usage low and constant.
        3. Efficient Watched Item Exclusion: Uses a pre-computed dictionary for fast lookups.
        4. Vectorized Top-K: Uses NumPy's argpartition for fast top-k selection across the entire batch.
        """
        if self.B is None:
            raise RuntimeError("The model has not been fitted yet. Call fit() first.")
            
        print(f"Generating top-{k} recommendations for {len(users)} users...")
        
        # --- Pre-computation for Watched Item Exclusion ---
        # This is much faster than doing groupby.apply(list)
        watched_items_dict = train_df.groupby('user_id')['encoded_item'].apply(list).to_dict()
        
        # Get the internal integer encodings for the users we want to predict for
        encoded_users_predict = self.user_enc.transform(users)
        
        all_recs = []
        # --- Batch Processing Loop ---
        for i in tqdm(range(0, len(encoded_users_predict), batch_size), desc="Predicting in Batches"):
            batch_user_indices = encoded_users_predict[i:i+batch_size]
            
            # --- Score Calculation (for the batch) ---
            # This is the core prediction step: (Users x Items) @ (Items x Items)
            batch_scores = self.X[batch_user_indices, :].dot(self.B)

            # --- Watched Item Exclusion (for the batch) ---
            for j, user_idx in enumerate(batch_user_indices):
                user_id = self.user_enc.inverse_transform([user_idx])[0]
                items_to_exclude = watched_items_dict.get(user_id, [])
                if items_to_exclude:
                    batch_scores[j, items_to_exclude] = -np.inf

            # --- Top-K Selection (for the batch) ---
            # np.argpartition is much faster than np.argsort for finding top-k
            top_k_indices = np.argpartition(batch_scores, -k, axis=1)[:, -k:]
            
            # We need both the scores and the original user indices for the results
            top_k_scores = np.take_along_axis(batch_scores, top_k_indices, axis=1)
            batch_user_ids = np.repeat(users[i:i+len(batch_user_indices)], k)
            
            # --- Formatting results for the batch ---
            batch_df = pd.DataFrame({
                'user_id': batch_user_ids,
                'item_id': self.item_enc.inverse_transform(top_k_indices.flatten()),
                'score': top_k_scores.flatten()
            })
            all_recs.append(batch_df)
            
        # --- Final Combination ---
        final_recs = pd.concat(all_recs)
        final_recs = final_recs.sort_values(['user_id', 'score'], ascending=[True, False])
        print("Prediction complete.")
        return final_recs

In [4]:
class FastEvaluator:
    def __init__(self, test_df, k):
        """
        Initializes the evaluator with ground truth data.
        
        Key Optimizations:
        1. Converts ground truth into a dictionary of sets for O(1) average time complexity lookups.
        2. Avoids slow pandas merge and groupby operations during metric calculation.
        """
        print("Initializing FastEvaluator...")
        self.k = k
        # Create the ground truth dictionary: {user_id: {item_id_1, item_id_2, ...}}
        self.ground_truth = test_df.groupby('user_id')['item_id'].apply(set).to_dict()
        print("Evaluator ready.")

    def evaluate(self, recommendations_df):
        print(f"Evaluating recommendations at k={self.k}...")
        # Group recommendations by user once
        recs_grouped = recommendations_df.groupby('user_id')['item_id'].apply(list)
        
        total_precision = 0
        total_recall = 0
        num_users = 0

        for user_id, recommended_items in recs_grouped.items():
            true_items = self.ground_truth.get(user_id)
            if true_items:
                num_users += 1
                # Use set intersection to find hits quickly
                hits = len(set(recommended_items[:self.k]).intersection(true_items))
                
                total_precision += hits / self.k
                total_recall += hits / len(true_items)
        
        avg_precision = total_precision / num_users if num_users > 0 else 0
        avg_recall = total_recall / num_users if num_users > 0 else 0
        
        print("--- Evaluation Results ---")
        print(f"Precision@{self.k}: {avg_precision:.4f}")
        print(f"Recall@{self.k}:    {avg_recall:.4f}")
        
        return {"precision": avg_precision, "recall": avg_recall}

In [5]:
# --- Train-Test Split ---
train_df, test_df = train_test_split(ratings_df, test_size=0.05, random_state=42)

# Ensure all users and items in the test set are also in the training set
test_df = test_df[test_df['user_id'].isin(train_df['user_id'])]
test_df = test_df[test_df['item_id'].isin(train_df['item_id'])]

In [6]:
# --- Instantiate and Train the Model ---
ease_model = EASE()
ease_model.fit(train_df, lambda_=250, implicit=False)

Fitting EASE model...


In [6]:
# --- 4. Generating Recommendations ---
test_users = test_df['user_id'].unique()
k = 10

start_time = time.time()
recommendations = ease_model.predict(train_df, test_users, k)
print(f"Total prediction time: {time.time() - start_time:.2f} seconds")


# --- 5. Fast Evaluation ---
start_time = time.time()
evaluator = FastEvaluator(test_df, k=k)
results = evaluator.evaluate(recommendations)
print(f"Total evaluation time: {time.time() - start_time:.2f} seconds")

Process SpawnPoolWorker-5:
  0%|          | 0/350 [00:00<?, ?it/s]Process SpawnPoolWorker-4:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/opt/anaconda3/lib/python3.11/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'EASE.predict_for_user' on <module '__main__' (built-in)>
  File "/opt/anaconda3/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kw

KeyboardInterrupt: 

In [ ]:
# --- Evaluation Metrics ---
def precision_at_k(recommendations, ground_truth, k):
    """Calculates precision@k."""
    merged = pd.merge(recommendations, ground_truth, on=['user_id', 'item_id'], how='left', indicator=True)
    hits = merged[merged['_merge'] == 'both'].groupby('user_id').size().reset_index(name='hits')
    precision = (hits['hits'] / k).mean()
    return precision

def recall_at_k(recommendations, ground_truth, k):
    """Calculates recall@k."""
    ground_truth_counts = ground_truth.groupby('user_id').size().reset_index(name='total_relevant')
    merged = pd.merge(recommendations, ground_truth, on=['user_id', 'item_id'], how='inner')
    hits = merged.groupby('user_id').size().reset_index(name='hits')
    
    merged_counts = pd.merge(hits, ground_truth_counts, on='user_id')
    recall = (merged_counts['hits'] / merged_counts['total_relevant']).mean()
    return recall

def map_at_k(recommendations, ground_truth, k):
    """Calculates Mean Average Precision@k."""
    ap_sum = 0
    # Group recommendations and ground truth by user
    rec_grouped = recommendations.groupby('user_id')
    gt_grouped = ground_truth.groupby('user_id')
    
    for user_id, group in rec_grouped:
        if user_id in gt_grouped.groups:
            relevant_items = set(gt_grouped.get_group(user_id)['item_id'])
            recommended_items = group['item_id'].tolist()
            
            hits = 0
            precision_sum = 0
            for i, item_id in enumerate(recommended_items):
                if item_id in relevant_items:
                    hits += 1
                    precision_sum += hits / (i + 1)
            
            ap_sum += precision_sum / min(len(relevant_items), k)
            
    return ap_sum / len(rec_grouped)

def ndcg_at_k(recommendations, ground_truth, k):
    """Calculates Normalized Discounted Cumulative Gain@k."""
    ndcg_sum = 0
    # Group recommendations and ground truth by user
    rec_grouped = recommendations.groupby('user_id')
    gt_grouped = ground_truth.groupby('user_id')
    
    for user_id, group in rec_grouped:
        if user_id in gt_grouped.groups:
            relevant_items = set(gt_grouped.get_group(user_id)['item_id'])
            recommended_items = group['item_id'].tolist()
            
            # Calculate DCG
            dcg = 0
            for i, item_id in enumerate(recommended_items):
                if item_id in relevant_items:
                    dcg += 1 / np.log2(i + 2) # +2 because ranks are 1-based
            
            # Calculate IDCG
            idcg = sum([1 / np.log2(i + 2) for i in range(min(len(relevant_items), k))])
            
            ndcg_sum += dcg / idcg if idcg > 0 else 0
            
    return ndcg_sum / len(rec_grouped)


# --- Run Evaluation ---
ground_truth = test_df[['user_id', 'item_id']]

precision = precision_at_k(recommendations, ground_truth, k)
recall = recall_at_k(recommendations, ground_truth, k)
map_k = map_at_k(recommendations, ground_truth, k)
ndcg = ndcg_at_k(recommendations, ground_truth, k)

# --- Display Results ---
print("--- EASE Model Evaluation ---")
print(f"Top-{k} Recommendations")
print(f"Precision@{k}: {precision:.4f}")
print(f"Recall@{k}:    {recall:.4f}")
print(f"MAP@{k}:       {map_k:.4f}")
print(f"NDCG@{k}:      {ndcg:.4f}")